# Crop Yield Prediction Using Machine Learning

## 1. Project framing and research question

**Objective:** build and evaluate regression models that estimate crop yield from crop type, season, state, cultivated area, annual rainfall, fertiliser and pesticide inputs.

**Research question:** how accurately can historical crop-yield records be used to predict yield in later years, and which regression approach performs best under time-ordered validation?

**Target:** `Yield` (continuous; regression task). **Excluded predictors:** `Production` and `Crop_Year`. Production is excluded because it is closely tied to yield and cultivated area, while the year is used to create chronological training, validation and hold-out periods rather than as a predictor.

**Evaluation design:** expanding-window validation within the training period, followed by a separate later-year hold-out test. Associations in the exploratory input analysis are descriptive, not causal.

In [ ]:
import os, glob, zipfile, warnings, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TRAIN_END_YEAR = 2015
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 2. Load the CSV file

In [ ]:
# Search for a CSV directly, or extract the first CSV found inside a ZIP file.
csv_candidates = glob.glob("*.csv") + glob.glob("/content/**/*.csv", recursive=True)
zip_candidates = glob.glob("*.zip") + glob.glob("/content/**/*.zip", recursive=True)

if csv_candidates:
    csv_path = csv_candidates[0]
    df = pd.read_csv(csv_path)
    print("Loaded:", csv_path)
elif zip_candidates:
    zip_path = zip_candidates[0]
    with zipfile.ZipFile(zip_path) as z:
        csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV file found inside the ZIP archive.")
        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f)
    print("Loaded", csv_names[0], "from", zip_path)
else:
    raise FileNotFoundError("Upload crop_yield.csv or its ZIP file to the notebook first.")

print("Shape:", df.shape)
display(df.head())
print("\\nColumns:", df.columns.tolist())

## 3. Clean and inspect the data

In [ ]:
# Standardise column names and categorical text.
df.columns = df.columns.str.strip()
required = [
    "Crop", "Crop_Year", "Season", "State", "Area",
    "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"
]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Required columns missing from dataset: {missing_cols}")

for c in ["Crop", "Season", "State"]:
    df[c] = df[c].astype("string").str.strip()

for c in ["Crop_Year", "Area", "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows and records without a usable target/year.
df = df.drop_duplicates()
df = df.dropna(subset=["Crop_Year", "Yield"])
df = df[df["Yield"] >= 0].copy()
df["Crop_Year"] = df["Crop_Year"].astype(int)

# Convert invalid negative input values to missing; the pipeline imputes them.
for c in ["Area", "Annual_Rainfall", "Fertilizer", "Pesticide"]:
    df.loc[df[c] < 0, c] = np.nan

print("Cleaned shape:", df.shape)
print("Year range:", df["Crop_Year"].min(), "to", df["Crop_Year"].max())
display(df.describe(include="all").T)

## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["Yield"], bins=60, ax=axes[0])
axes[0].set_title("Yield distribution (raw)")
sns.histplot(np.log1p(df["Yield"]), bins=60, ax=axes[1])
axes[1].set_title("Yield distribution (log1p)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
yearly = df.groupby("Crop_Year")["Yield"].median()
yearly.plot(marker="o")
plt.title("Median crop yield by year")
plt.ylabel("Median yield")
plt.show()

display(df.groupby("Season")["Yield"].agg(["count", "mean", "median"]).sort_values("median", ascending=False))

# Additional EDA: numeric relationships and crop-level target summaries.
numeric_eda = df[["Area", "Production", "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"]]
plt.figure(figsize=(9, 7))
sns.heatmap(numeric_eda.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix of numeric dataset columns")
plt.tight_layout()
plt.show()

crop_summary = (
    df.groupby("Crop")["Yield"]
      .agg(records="size", mean_yield="mean", median_yield="median")
      .sort_values("median_yield", ascending=False)
)
display(crop_summary.head(15))


## 5. Define the time-based train/test split

All records through 2015 are used for training. Records from 2016 onward are held out for testing. This is a more realistic test of forecasting into later years than a random split.

The features follow the research question: area, annual rainfall, fertiliser, pesticide, crop, season and state. `Production` and `Crop_Year` are not predictors.

In [ ]:
feature_cols = [
    "Area", "Annual_Rainfall", "Fertilizer", "Pesticide",
    "Crop", "Season", "State"
]
numeric_features = ["Area", "Annual_Rainfall", "Fertilizer", "Pesticide"]
categorical_features = ["Crop", "Season", "State"]

train_df = df[df["Crop_Year"] <= TRAIN_END_YEAR].copy()
test_df = df[df["Crop_Year"] > TRAIN_END_YEAR].copy()

if train_df.empty or test_df.empty:
    raise ValueError(
        f"Time split produced an empty set. Dataset years: "
        f"{df['Crop_Year'].min()}–{df['Crop_Year'].max()}. "
        "Adjust TRAIN_END_YEAR if needed."
    )

X_train, X_test = train_df[feature_cols], test_df[feature_cols]
y_train = np.log1p(train_df["Yield"].values)
y_test = test_df["Yield"].values

print(f"Train: {len(train_df):,} rows, years {train_df.Crop_Year.min()}–{train_df.Crop_Year.max()}")
print(f"Test : {len(test_df):,} rows, years {test_df.Crop_Year.min()}–{test_df.Crop_Year.max()}")

## 6. Preprocessing and outlier treatment

A quantile clipper learns the 1st and 99th percentile limits from the **training set only**, then applies those limits to both train and test data. This reduces the influence of extreme input values without allowing test data to influence preprocessing.

Numeric variables are median-imputed and standardised. Categorical variables are mode-imputed and one-hot encoded. Polynomial features are added only to numeric variables for the polynomial model.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", QuantileClipper(0.01, 0.99)),
    ("scaler", StandardScaler())
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocess, numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

# Polynomial expansion applies to numeric inputs only; categorical features remain one-hot encoded.
polynomial_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clipper", QuantileClipper(0.01, 0.99)),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

## 7. Train and compare regression models

In [ ]:
models = {
    "Dummy Baseline (median)": Pipeline([
        ("preprocess", preprocessor),
        ("model", DummyRegressor(strategy="median"))
    ]),
    "Multiple Linear Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),
    "Polynomial Regression (degree 2)": Pipeline([
        ("preprocess", polynomial_preprocessor),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=10.0))
    ]),
    "Lasso Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", Lasso(alpha=0.001, max_iter=20000))
    ]),
    "Decision Tree": Pipeline([
        ("preprocess", preprocessor),
        ("model", DecisionTreeRegressor(
            max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE
        ))
    ]),
    "k-NN Regressor": Pipeline([
        ("preprocess", preprocessor),
        ("model", KNeighborsRegressor(n_neighbors=10, weights="distance"))
    ])
}

results = []
fitted_models = {}
predictions = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred_log = pipe.predict(X_test)
    pred_yield = np.maximum(0, np.expm1(pred_log))
    predictions[name] = pred_yield
    fitted_models[name] = pipe

    rmse = np.sqrt(mean_squared_error(y_test, pred_yield))
    mae = mean_absolute_error(y_test, pred_yield)
    r2 = r2_score(y_test, pred_yield)
    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

## 8. Time-based cross-validation and hyperparameter tuning

Model selection is performed using expanding-window validation folds built only from the training years. Each validation fold contains later years than its corresponding training fold. The final 2016–2020 hold-out set is not used to choose the model or tune its parameters.

The custom scorer converts predictions back from log1p scale and calculates MAE in the original yield units. A Decision Tree is tuned with GridSearchCV; its best estimator is compared with the baseline and the other candidate pipelines.

In [ ]:
# Build expanding-window folds using training years only.
train_years = train_df["Crop_Year"].to_numpy()
temporal_cv_splits = []

for train_end, valid_start, valid_end in [
    (2007, 2008, 2010),
    (2010, 2011, 2013),
    (2013, 2014, 2015)
]:
    train_idx = np.flatnonzero(train_years <= train_end)
    valid_idx = np.flatnonzero(
        (train_years >= valid_start) & (train_years <= valid_end)
    )
    if len(train_idx) == 0 or len(valid_idx) == 0:
        raise ValueError(
            f"Empty temporal CV fold: train <= {train_end}, "
            f"validation {valid_start}-{valid_end}. Check dataset years."
        )
    temporal_cv_splits.append((train_idx, valid_idx))

print("Temporal CV folds (train years -> validation years):")
for train_idx, valid_idx in temporal_cv_splits:
    print(
        f"{train_years[train_idx].min()}-{train_years[train_idx].max()} -> "
        f"{train_years[valid_idx].min()}-{train_years[valid_idx].max()} "
        f"({len(train_idx):,} train, {len(valid_idx):,} validation rows)"
    )

# Scikit-learn scorers are maximised, so return negative MAE.
# Both y and estimator predictions are on log1p scale.
def yield_mae_scorer(estimator, X, y):
    actual_yield = np.expm1(np.asarray(y))
    predicted_yield = np.maximum(0, np.expm1(estimator.predict(X)))
    return -mean_absolute_error(actual_yield, predicted_yield)

cv_rows = []
for name, estimator in models.items():
    fold_scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        cv=temporal_cv_splits,
        scoring=yield_mae_scorer,
        n_jobs=-1
    )
    cv_rows.append({
        "Model": name,
        "CV_MAE_mean": -fold_scores.mean(),
        "CV_MAE_std": fold_scores.std()
    })

# Tune a Decision Tree using only the temporal CV folds.
tree_search = GridSearchCV(
    estimator=models["Decision Tree"],
    param_grid={
        "model__max_depth": [8, 12, None],
        "model__min_samples_leaf": [3, 5, 10],
        "model__min_samples_split": [2, 10]
    },
    scoring=yield_mae_scorer,
    cv=temporal_cv_splits,
    n_jobs=-1,
    refit=True
)
tree_search.fit(X_train, y_train)

tuned_tree = tree_search.best_estimator_
tuned_name = "Tuned Decision Tree"
fitted_models[tuned_name] = tuned_tree
tuned_pred = np.maximum(0, np.expm1(tuned_tree.predict(X_test)))
predictions[tuned_name] = tuned_pred

cv_rows.append({
    "Model": tuned_name,
    "CV_MAE_mean": -tree_search.best_score_,
    "CV_MAE_std": tree_search.cv_results_["std_test_score"][tree_search.best_index_]
})

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_MAE_mean").reset_index(drop=True)
display(cv_results_df)

print("Best Decision Tree parameters:", tree_search.best_params_)
print("Selected model by temporal CV:", cv_results_df.iloc[0]["Model"])

# Add the tuned model's final hold-out metrics for reporting only.
results.append({
    "Model": tuned_name,
    "RMSE": np.sqrt(mean_squared_error(y_test, tuned_pred)),
    "MAE": mean_absolute_error(y_test, tuned_pred),
    "R2": r2_score(y_test, tuned_pred)
})
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

## 9. Final hold-out performance comparison

The table and plots below report performance on the untouched 2016–2020 test period. They are for final reporting only; model selection was made using training-period temporal cross-validation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for metric, ax in zip(["RMSE", "MAE", "R2"], axes):
    sns.barplot(data=results_df, x=metric, y="Model", ax=ax)
    ax.set_title(metric + " on 2016–2020 test set")
plt.tight_layout()
plt.show()

best_name = cv_results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = predictions[best_name]
print("Model selected using temporal CV:", best_name)
print("Final hold-out metrics:")
display(results_df[results_df["Model"] == best_name])

## 10. Actual vs predicted yield

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_pred, alpha=0.35)
max_val = max(np.nanmax(y_test), np.nanmax(best_pred))
plt.plot([0, max_val], [0, max_val], linestyle="--")
plt.xlabel("Actual yield")
plt.ylabel("Predicted yield")
plt.title(f"Actual vs predicted yield — {best_name}")
plt.tight_layout()
plt.show()

residuals = y_test - best_pred
plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Prediction residuals (actual − predicted)")
plt.xlabel("Residual")
plt.show()

## 11. Do higher fertiliser and pesticide inputs buy higher yield?

This section examines associations, not causal effects. Raw total input values can be misleading because larger cultivated areas tend to use more inputs. We therefore calculate input intensity per unit area and compare it with yield. The analysis is descriptive and does not establish that changing an input will cause yield to change.

In [ ]:
eff = df.copy()
eff = eff[(eff["Area"] > 0) & eff["Yield"].notna()].copy()
eff["Fertilizer_per_area"] = eff["Fertilizer"] / eff["Area"]
eff["Pesticide_per_area"] = eff["Pesticide"] / eff["Area"]

display(eff[["Yield", "Fertilizer_per_area", "Pesticide_per_area"]].corr())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=eff, x="Fertilizer_per_area", y="Yield",
            scatter_kws={"alpha": 0.15, "s": 12}, line_kws={"color": "red"},
            ax=axes[0], lowess=True)
axes[0].set_title("Yield vs fertiliser per area")
sns.regplot(data=eff, x="Pesticide_per_area", y="Yield",
            scatter_kws={"alpha": 0.15, "s": 12}, line_kws={"color": "red"},
            ax=axes[1], lowess=True)
axes[1].set_title("Yield vs pesticide per area")
plt.tight_layout()
plt.show()

# Summarise input intensity and yield by intensity quintile.
for input_col in ["Fertilizer_per_area", "Pesticide_per_area"]:
    eff[input_col + "_group"] = pd.qcut(
        eff[input_col].rank(method="first"), 5,
        labels=["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (highest)"]
    )
    summary = eff.groupby(input_col + "_group", observed=False)["Yield"].agg(
        ["count", "mean", "median"]
    )
    print("\\n", input_col)
    display(summary)

## 12. State-level input-use efficiency patterns

A simple descriptive indicator is median yield divided by median input intensity. It is not a complete agronomic efficiency measure and should not be interpreted causally. The table helps identify states for further investigation.

In [ ]:
state_efficiency = (
    eff.groupby("State")
       .agg(
           records=("Yield", "size"),
           median_yield=("Yield", "median"),
           median_fertiliser_per_area=("Fertilizer_per_area", "median"),
           median_pesticide_per_area=("Pesticide_per_area", "median")
       )
       .query("records >= 30")
       .sort_values("median_yield", ascending=False)
)
display(state_efficiency.head(20))

fig, ax = plt.subplots(figsize=(11, 7))
top_states = state_efficiency.head(15).sort_values("median_yield")
top_states["median_yield"].plot(kind="barh", ax=ax)
ax.set_title("Top 15 states by median crop yield (at least 30 records)")
ax.set_xlabel("Median yield")
plt.tight_layout()
plt.show()

## 13. Permutation feature importance

Permutation importance estimates how much test-set performance worsens when a feature is shuffled. This is calculated on the best model using the held-out years. Because the model uses a preprocessing pipeline, importance is computed on the original input columns.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance_mean": perm.importances_mean,
    "Importance_std": perm.importances_std
}).sort_values("Importance_mean", ascending=False)

display(importance_df)
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="Importance_mean", y="Feature")
plt.title("Permutation importance on 2016–2020 hold-out")
plt.xlabel("Increase in negative MAE when shuffled")
plt.tight_layout()
plt.show()

## 14. Save predictions and model comparison

In [ ]:
prediction_output = test_df[["Crop_Year", "Crop", "Season", "State", "Yield"]].copy()
prediction_output["Predicted_Yield"] = best_pred
prediction_output["Absolute_Error"] = np.abs(
    prediction_output["Yield"] - prediction_output["Predicted_Yield"]
)
prediction_output.to_csv("crop_yield_test_predictions.csv", index=False)
results_df.to_csv("crop_yield_model_comparison.csv", index=False)
cv_results_df.to_csv("crop_yield_temporal_cv_results.csv", index=False)

# Save a reproducibility record for this exact run and dataset.
with open(csv_path, "rb") as dataset_file:
    dataset_sha256 = hashlib.sha256(dataset_file.read()).hexdigest()

run_metadata = {
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "random_state": RANDOM_STATE,
    "dataset_file": os.path.basename(csv_path),
    "dataset_sha256": dataset_sha256,
    "rows_after_cleaning": int(len(df)),
    "columns": df.columns.tolist(),
    "year_min": int(df["Crop_Year"].min()),
    "year_max": int(df["Crop_Year"].max()),
    "train_end_year": TRAIN_END_YEAR,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "features": feature_cols,
    "target": "Yield",
    "model_selected_by_temporal_cv": str(best_name)
}
with open("crop_yield_run_metadata.json", "w", encoding="utf-8") as metadata_file:
    json.dump(run_metadata, metadata_file, indent=2)

print("Saved:")
print("- crop_yield_test_predictions.csv")
print("- crop_yield_model_comparison.csv")
print("- crop_yield_temporal_cv_results.csv")
print("- crop_yield_run_metadata.json")
display(prediction_output.head())